# Offline Patch Optimization

This notebook selects and validates the offline patch representation used for deployment.

Main flow:
- prepare MVTec AD train, validation and test images
- keep YOLO26 frozen and extract local feature maps
- compare image size, feature depth, patch grid and top-patch fraction
- choose one shared configuration
- validate all 15 categories
- export clean `patch_memory_bank.pt` and `threshold.json` files.


In [12]:
# Install only packages missing from the current runtime.
import sys, subprocess, importlib.util

def install_if_missing(module_name, pip_name=None):
    """Install a package only when its import is unavailable."""
    pip_name = pip_name or module_name
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

install_if_missing("ultralytics")
install_if_missing("sklearn", "scikit-learn")
install_if_missing("pandas")


In [13]:
# Setup/Define neccessary need.
import json
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
from ultralytics import YOLO

# Keep data split and memory sampling repeatable.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Use GPU when available else CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cuda


In [14]:
# Dataset paths and experiment settings.
MVTec_ROOT = Path("/kaggle/input/datasets/ipythonx/mvtec-ad")
WORK_ROOT = Path("/kaggle/working/mvtec_yolo26_ttl_rev18")
REPORT_ROOT = WORK_ROOT / "reports"
EXPORT_ROOT = WORK_ROOT / "deploy_candidate"

CATEGORIES = [
    "bottle", "cable", "capsule", "carpet", "grid",
    "hazelnut", "leather", "metal_nut", "pill", "screw",
    "tile", "toothbrush", "transistor", "wood", "zipper",
]

# Smaller group used to compare candidate patch settings.
FOCUS_CATEGORIES = ["grid", "carpet", "leather", "screw", "pill"]

MODEL_NAME = "yolo26n-cls.pt"
BATCH_SIZE = 12
NUM_WORKERS = 2 if DEVICE == "cuda" else 0
TRAIN_VAL_RATIO = 0.20
OFFLINE_THRESHOLD_QUANTILE = 0.995

# Candidate values tested before full validation.
IMAGE_SIZES = [224, 384, 512]
PATCH_GRIDS = [8, 14, 20]
PATCH_TOP_FRACTIONS = [0.01, 0.03, 0.05, 0.10]

# Local feature choices refer to the most recent usable 4D YOLO feature maps.
FEATURE_CHOICES = {
    "last1": [-1],
    "last2": [-2, -1],
    "last3": [-3, -2, -1],
}

PATCH_MEMORY_MAX = 16000

# Final candidate. Update only after reviewing the ablation.
FINAL_IMAGE_SIZE = 384
FINAL_PATCH_GRID = 14
FINAL_PATCH_TOP_FRACTION = 0.05
FINAL_FEATURE_CHOICE = "last2"

REPORT_ROOT.mkdir(parents=True, exist_ok=True)
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

assert MVTec_ROOT.exists(), f"Dataset not found: {MVTec_ROOT}"


In [15]:
# Prepare one category into train-normal, validation-normal and test folders.
def reset_dir(path: Path):
    """Recreate a folder so prepared data starts clean."""
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def save_as_jpg(src_path: Path, dst_path: Path, size: int):
    """Convert to RGB, resize and save a consistent JPG copy."""
    img = Image.open(src_path).convert("RGB")
    img = img.resize((size, size), Image.Resampling.BILINEAR)
    img.save(dst_path, quality=95)

def prepare_category(category: str, img_size: int):
    """Prepare normal train/validation data and normal/anomaly test data."""
    src_root = MVTec_ROOT / category
    out_root = WORK_ROOT / "prepared" / f"{category}_{img_size}"

    train_dir = out_root / "train" / "normal"
    val_dir = out_root / "val" / "normal"
    test_good_dir = out_root / "test" / "normal"
    test_bad_dir = out_root / "test" / "anomaly"

    for d in [train_dir, val_dir, test_good_dir, test_bad_dir]:
        reset_dir(d)

# Fixed-seed split keeps the same train/validation samples on repeated runs.
    train_good = sorted((src_root / "train" / "good").glob("*"))
    rng = random.Random(SEED)
    idx = list(range(len(train_good)))
    rng.shuffle(idx)
    split = int(len(idx) * (1.0 - TRAIN_VAL_RATIO))
    train_idx = set(idx[:split])

    for i, src in enumerate(train_good):
        dst = train_dir if i in train_idx else val_dir
        save_as_jpg(src, dst / f"{src.stem}.jpg", img_size)

    # test/good is normal, every other test folder is treated as anomaly.
    for defect_dir in sorted((src_root / "test").iterdir()):
        if not defect_dir.is_dir():
            continue
        dst = test_good_dir if defect_dir.name == "good" else test_bad_dir
        for src in sorted(defect_dir.glob("*")):
            save_as_jpg(src, dst / f"{defect_dir.name}_{src.stem}.jpg", img_size)

    return train_dir, val_dir, test_good_dir, test_bad_dir


In [16]:
# PyTorch dataset and loader for prepared JPG files.
class ImageDataset(Dataset):
    """Load prepared images as normalized CHW float tensors."""
    def __init__(self, paths, labels):
        self.paths = list(paths)
        self.labels = list(labels)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")
        x = np.asarray(img, dtype=np.float32) / 255.0
        x = torch.from_numpy(np.transpose(x, (2, 0, 1))).float()
        return x, int(self.labels[idx]), str(path)

def make_loader(normal_dir: Path, anomaly_dir=None):
    """Build a deterministic loader with normal=0 and anomaly=1 labels."""
    paths = sorted(normal_dir.glob("*.jpg"))
    labels = [0] * len(paths)

    if anomaly_dir is not None:
        bad = sorted(anomaly_dir.glob("*.jpg"))
        paths += bad
        labels += [1] * len(bad)

    ds = ImageDataset(paths, labels)
    dl = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
    )
    return dl


In [17]:
# Frozen YOLO26 feature extractor for local patch embeddings.
class YOLO26LocalExtractor(nn.Module):
    """Capture selected local YOLO feature maps without training YOLO."""
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.yolo = YOLO(model_name)
        self.core = self.yolo.model.to(DEVICE).eval()

        # YOLO remains fixed throughout offline preparation.
        for p in self.core.parameters():
            p.requires_grad = False

        self.local_cache = []
        self.hooks = []

        # Hooks read recent 4D feature maps without changing the forward pass.
        for layer in list(self.core.model)[-12:]:
            self.hooks.append(layer.register_forward_hook(self._hook))

    def _hook(self, module, inputs, output):
        """Store usable spatial feature maps from the current forward pass."""
        if torch.is_tensor(output) and output.ndim == 4:
            if min(output.shape[-2:]) >= 4:
                self.local_cache.append(output)

    @torch.no_grad()
    def extract(self, x, patch_grid=14, feature_indices=(-2, -1)):
        """Resize selected maps to one grid, concatenate and L2-normalize."""
        self.local_cache = []
        _ = self.core(x)

        if len(self.local_cache) < max(abs(i) for i in feature_indices):
            raise RuntimeError(
                f"Not enough usable feature maps: {len(self.local_cache)}"
            )

        selected = [self.local_cache[i] for i in feature_indices]
        parts = []

        for fmap in selected:
            fmap = F.interpolate(
                fmap,
                size=(patch_grid, patch_grid),
                mode="bilinear",
                align_corners=False,
            )
            fmap = fmap.permute(0, 2, 3, 1)
            parts.append(fmap)

        patch = torch.cat(parts, dim=-1)
        return F.normalize(patch, dim=-1)

extractor = YOLO26LocalExtractor()
print("Extractor ready")


Extractor ready


In [18]:
# Convert images to patches, build normal memory and calculate anomaly scores.
@torch.no_grad()
def extract_patches(loader, patch_grid, feature_indices):
    """Extract patch embeddings, labels and file paths for one loader."""
    patches, labels, paths = [], [], []

    for x, y, p in loader:
        x = x.to(DEVICE, non_blocking=True)
        z = extractor.extract(
            x,
            patch_grid=patch_grid,
            feature_indices=feature_indices,
        )
        patches.append(z.cpu())
        labels.extend(np.asarray(y).astype(int).tolist())
        paths.extend(list(p))

    return torch.cat(patches, dim=0), np.asarray(labels, dtype=int), paths

def sample_patch_memory(patches, max_patches=PATCH_MEMORY_MAX, seed=SEED):
    """Flatten normal patches and keep a repeatable subset if memory is large."""
    bank = patches.reshape(-1, patches.shape[-1]).float()
    bank = F.normalize(bank, dim=1)

    if len(bank) <= max_patches:
        return bank

    rng = np.random.default_rng(seed)
    idx = rng.choice(len(bank), size=max_patches, replace=False)
    return bank[idx]

def patch_score(patch_emb, patch_bank, top_fraction=0.05):
    """Score each image from its highest nearest-normal patch distances."""
    bank = F.normalize(patch_bank.float(), dim=1)
    scores = []

    for image_patches in patch_emb:
        q = image_patches.reshape(-1, image_patches.shape[-1]).float()
        q = F.normalize(q, dim=1)

        # Nearest normal patch for each query patch.
        # Maximum cosine similarity gives the nearest normal patch.
        sim = q @ bank.T
        dist = 1.0 - sim.max(dim=1).values

        # Use the most anomalous patch subset.
        # Average only the most abnormal patch subset.
        k = max(1, int(np.ceil(len(dist) * top_fraction)))
        score = torch.topk(dist, k=k, largest=True).values.mean()
        scores.append(float(score))

    return np.asarray(scores, dtype=np.float32)

def threshold_from_normal(scores):
    """Set the offline threshold from normal validation scores only."""
    return float(np.quantile(np.asarray(scores), OFFLINE_THRESHOLD_QUANTILE))

def evaluate(labels, scores, threshold):
    """Calculate AUROC, accuracy, class recall and Macro F1."""
    pred = (scores > threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(
        labels, pred, labels=[0, 1], zero_division=0
    )
    return {
        "auroc": roc_auc_score(labels, scores),
        "accuracy": accuracy_score(labels, pred),
        "normal_recall": r[0],
        "anomaly_recall": r[1],
        "macro_f1": float(np.mean(f1)),
    }


In [19]:
# Compare candidate patch settings on the focus categories.
rows = []

for category in FOCUS_CATEGORIES:
    for img_size in IMAGE_SIZES:
        train_dir, val_dir, good_dir, bad_dir = prepare_category(category, img_size)

        train_loader = make_loader(train_dir)
        val_loader = make_loader(val_dir)
        test_loader = make_loader(good_dir, bad_dir)

        for feature_name, feature_indices in FEATURE_CHOICES.items():
            for patch_grid in PATCH_GRIDS:
                # Extract once for this feature/grid setting and reuse across top fractions.
                train_patch, _, _ = extract_patches(
                    train_loader, patch_grid, feature_indices
                )
                val_patch, _, _ = extract_patches(
                    val_loader, patch_grid, feature_indices
                )
                test_patch, test_labels, _ = extract_patches(
                    test_loader, patch_grid, feature_indices
                )

                # Training-normal patches form the reference memory.
                bank = sample_patch_memory(train_patch)

                for top_fraction in PATCH_TOP_FRACTIONS:
                    val_scores = patch_score(
                        val_patch, bank, top_fraction=top_fraction
                    )
                    test_scores = patch_score(
                        test_patch, bank, top_fraction=top_fraction
                    )

                    threshold = threshold_from_normal(val_scores)
                    m = evaluate(test_labels, test_scores, threshold)

                    rows.append({
                        "category": category,
                        "img_size": img_size,
                        "feature_choice": feature_name,
                        "patch_grid": patch_grid,
                        "patch_top_fraction": top_fraction,
                        "threshold": threshold,
                        **m,
                    })

                    print(
                        category, img_size, feature_name,
                        patch_grid, top_fraction,
                        "AUROC=", round(m["auroc"], 4)
                    )

ablation_df = pd.DataFrame(rows)
ablation_df.to_csv(
    REPORT_ROOT / "rev18_weak_category_ablation.csv",
    index=False
)


grid 224 last1 8 0.01 AUROC= 0.5681
grid 224 last1 8 0.03 AUROC= 0.5806
grid 224 last1 8 0.05 AUROC= 0.6099
grid 224 last1 8 0.1 AUROC= 0.6241
grid 224 last1 14 0.01 AUROC= 0.584
grid 224 last1 14 0.03 AUROC= 0.604
grid 224 last1 14 0.05 AUROC= 0.6174
grid 224 last1 14 0.1 AUROC= 0.6274
grid 224 last1 20 0.01 AUROC= 0.6241
grid 224 last1 20 0.03 AUROC= 0.6149
grid 224 last1 20 0.05 AUROC= 0.6374
grid 224 last1 20 0.1 AUROC= 0.6366
grid 224 last2 8 0.01 AUROC= 0.6433
grid 224 last2 8 0.03 AUROC= 0.6792
grid 224 last2 8 0.05 AUROC= 0.7043
grid 224 last2 8 0.1 AUROC= 0.7084
grid 224 last2 14 0.01 AUROC= 0.6583
grid 224 last2 14 0.03 AUROC= 0.6792
grid 224 last2 14 0.05 AUROC= 0.6884
grid 224 last2 14 0.1 AUROC= 0.6984
grid 224 last2 20 0.01 AUROC= 0.6759
grid 224 last2 20 0.03 AUROC= 0.6984
grid 224 last2 20 0.05 AUROC= 0.7068
grid 224 last2 20 0.1 AUROC= 0.6951
grid 224 last3 8 0.01 AUROC= 0.6099
grid 224 last3 8 0.03 AUROC= 0.6675
grid 224 last3 8 0.05 AUROC= 0.6892
grid 224 last3 8 0.1

In [20]:
# Show the strongest setting per focus category, this is diagnostic only.
best_by_category = (
    ablation_df.sort_values(["category", "auroc"], ascending=[True, False])
    .groupby("category", as_index=False)
    .first()
)

display(best_by_category[[
    "category",
    "img_size",
    "feature_choice",
    "patch_grid",
    "patch_top_fraction",
    "auroc",
    "macro_f1",
    "normal_recall",
    "anomaly_recall",
]])

best_by_category.to_csv(
    REPORT_ROOT / "rev18_best_by_weak_category.csv",
    index=False
)


,category,img_size,feature_choice,patch_grid,patch_top_fraction,auroc,macro_f1,normal_recall,anomaly_recall
0,carpet,512,last2,8,0.01,0.968299,0.793651,0.535714,0.977528
1,grid,512,last3,20,0.03,0.914787,0.793962,0.857143,0.807018
2,leather,224,last1,20,0.01,0.972486,0.896770,0.875000,0.934783
3,pill,512,last1,20,0.05,0.954719,0.557788,1.000000,0.517730
4,screw,512,last2,20,0.05,0.926624,0.324514,1.000000,0.117647


In [21]:
# Rank one shared configuration using mean and minimum category AUROC.
fixed_summary = (
    ablation_df.groupby(
        ["img_size", "feature_choice", "patch_grid", "patch_top_fraction"]
    )
    .agg(
        mean_auroc=("auroc", "mean"),
        min_auroc=("auroc", "min"),
        mean_macro_f1=("macro_f1", "mean"),
    )
    .reset_index()
    .sort_values(
        ["mean_auroc", "min_auroc"],
        ascending=[False, False]
    )
)

display(fixed_summary.head(20))
fixed_summary.to_csv(
    REPORT_ROOT / "rev18_fixed_config_summary.csv",
    index=False
)


,img_size,feature_choice,patch_grid,patch_top_fraction,mean_auroc,min_auroc,mean_macro_f1
93,512,last2,20,0.03,0.923114,0.881370,0.586081
94,512,last2,20,0.05,0.919612,0.874687,0.593692
105,512,last3,20,0.03,0.919254,0.900815,0.620228
84,512,last2,8,0.01,0.918823,0.892231,0.615974
85,512,last2,8,0.03,0.917561,0.889724,0.618992
106,512,last3,20,0.05,0.912619,0.890285,0.608326
81,512,last1,20,0.03,0.912385,0.801170,0.610730
86,512,last2,8,0.05,0.912348,0.892663,0.588124
82,512,last1,20,0.05,0.911268,0.797828,0.614907
92,512,last2,20,0.01,0.909696,0.837928,0.589348


## Configuration selection

Use one shared setting rather than category-specific tuning.

Selection priority:
1. high mean AUROC across the focus categories
2. acceptable minimum-category AUROC
3. stable performance when the same setting is applied to all 15 categories.


In [22]:
# Validate the selected shared configuration on all 15 categories.
FINAL_IMAGE_SIZE = 384
FINAL_FEATURE_CHOICE = "last2"
FINAL_PATCH_GRID = 14
FINAL_PATCH_TOP_FRACTION = 0.05

final_rows = []

for category in CATEGORIES:
    train_dir, val_dir, good_dir, bad_dir = prepare_category(
        category, FINAL_IMAGE_SIZE
    )

    train_loader = make_loader(train_dir)
    val_loader = make_loader(val_dir)
    test_loader = make_loader(good_dir, bad_dir)

    feature_indices = FEATURE_CHOICES[FINAL_FEATURE_CHOICE]

    train_patch, _, _ = extract_patches(
        train_loader, FINAL_PATCH_GRID, feature_indices
    )
    val_patch, _, _ = extract_patches(
        val_loader, FINAL_PATCH_GRID, feature_indices
    )
    test_patch, test_labels, _ = extract_patches(
        test_loader, FINAL_PATCH_GRID, feature_indices
    )

    # Build reference memory from training-normal patches only.
    bank = sample_patch_memory(train_patch)

    val_scores = patch_score(
        val_patch, bank, FINAL_PATCH_TOP_FRACTION
    )
    test_scores = patch_score(
        test_patch, bank, FINAL_PATCH_TOP_FRACTION
    )

    # Normal validation scores set the offline threshold.
    threshold = threshold_from_normal(val_scores)
    m = evaluate(test_labels, test_scores, threshold)

    final_rows.append({
        "category": category,
        "threshold": threshold,
        **m,
    })

final_df = pd.DataFrame(final_rows)
display(final_df.sort_values("auroc"))

print("Mean AUROC:", final_df["auroc"].mean())
print("Mean Macro F1:", final_df["macro_f1"].mean())

final_df.to_csv(
    REPORT_ROOT / "rev18_final_all_categories.csv",
    index=False
)


,category,threshold,auroc,accuracy,normal_recall,anomaly_recall,macro_f1
4,grid,0.054524,0.824561,0.653846,0.952381,0.543860,0.646822
9,screw,0.080889,0.853249,0.312500,1.000000,0.075630,0.283854
11,toothbrush,0.054081,0.894444,0.785714,0.750000,0.800000,0.754386
8,pill,0.062283,0.921713,0.700599,1.000000,0.645390,0.647143
6,leather,0.025657,0.921875,0.790323,0.250000,0.978261,0.627369
3,carpet,0.026402,0.924157,0.837607,0.535714,0.932584,0.754771
2,capsule,0.032200,0.937375,0.818182,0.869565,0.807339,0.752500
14,zipper,0.024299,0.973739,0.933775,0.843750,0.957983,0.900867
13,wood,0.045799,0.979825,0.898734,0.684211,0.966667,0.850095
7,metal_nut,0.079963,0.980450,0.904348,1.000000,0.881720,0.868571


Mean AUROC: 0.9449283729822591
Mean Macro F1: 0.7842128409059874


In [23]:
# Export the clean offline patch memory and scoring config for each category.
feature_indices = FEATURE_CHOICES[FINAL_FEATURE_CHOICE]

for category in CATEGORIES:
    train_dir, val_dir, _, _ = prepare_category(
        category, FINAL_IMAGE_SIZE
    )

    train_loader = make_loader(train_dir)
    val_loader = make_loader(val_dir)

    train_patch, _, _ = extract_patches(
        train_loader, FINAL_PATCH_GRID, feature_indices
    )
    val_patch, _, _ = extract_patches(
        val_loader, FINAL_PATCH_GRID, feature_indices
    )

    bank = sample_patch_memory(train_patch)
    val_scores = patch_score(
        val_patch, bank, FINAL_PATCH_TOP_FRACTION
    )
    threshold = threshold_from_normal(val_scores)

    cat_dir = EXPORT_ROOT / category
    cat_dir.mkdir(parents=True, exist_ok=True)

    # Deployment memory contains training-normal patches only.
    torch.save(bank.cpu(), cat_dir / "patch_memory_bank.pt")

    # Keep scoring settings beside the memory for deployment.
    cfg = {
        "category": category,
        "threshold": threshold,
        "img_size": FINAL_IMAGE_SIZE,
        "score_method": "patch_nearest_normal",
        "feature_choice": FINAL_FEATURE_CHOICE,
        "patch_grid": FINAL_PATCH_GRID,
        "patch_top_fraction": FINAL_PATCH_TOP_FRACTION,
        "offline_threshold_quantile": OFFLINE_THRESHOLD_QUANTILE,
        "export_state": "clean_train_normal_only",
    }

    with open(cat_dir / "threshold.json", "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=4)

print("Reports:", REPORT_ROOT)
print("Candidate export:", EXPORT_ROOT)


Reports: /kaggle/working/mvtec_yolo26_ttl_rev18/reports
Candidate export: /kaggle/working/mvtec_yolo26_ttl_rev18/deploy_candidate


In [24]:
# Output locations for review and deployment copy.
print("Configuration summary:")
print(REPORT_ROOT / "rev18_fixed_config_summary.csv")
print()
print("All-category result:")
print(REPORT_ROOT / "rev18_final_all_categories.csv")
print()
print("Deployment export:")
print(EXPORT_ROOT)


Configuration summary:
/kaggle/working/mvtec_yolo26_ttl_rev18/reports/rev18_fixed_config_summary.csv

All-category result:
/kaggle/working/mvtec_yolo26_ttl_rev18/reports/rev18_final_all_categories.csv

Deployment export:
/kaggle/working/mvtec_yolo26_ttl_rev18/deploy_candidate
